
create surprise environment first 

### For local installation
### run this commands in anaconda prompt

conda env list

conda create -n surprise python=3.10

conda activate surprise

pip install numpy==1.26.4

conda install -c conda-forge scikit-surprise

conda install -c conda-forge pandas 


open anaonda navigator -> jupyter notebook

 

In [7]:
import surprise
import pandas as pd
import numpy as np

import os
os.chdir(r'E:\PGCP-AI\PML\Cases\SURPRISE\filmtrust') 
'''  
Surprise is a Python scikit for building and analyzing recommender systems that deal with explicit rating data.

In the context of Python and recommender systems, Surprise (often imported as surprise or scikit-surprise) is a popular,
open-source Python scikit library specifically designed for building and analyzing recommender systems—particularly collaborative 
filtering algorithms.
'''

'  \nSurprise is a Python scikit for building and analyzing recommender systems that deal with explicit rating data.\n\nIn the context of Python and recommender systems, Surprise (often imported as surprise or scikit-surprise) is a popular,\nopen-source Python scikit library specifically designed for building and analyzing recommender systems—particularly collaborative \nfiltering algorithms.\n'

In [8]:
ratings = pd.read_csv('ratings.txt' , sep=' ', names= ['uid','iid','rating'])
ratings.head()  # uesrid    itemid   rating

,uid,iid,rating
0,1,1,2.0
1,1,2,4.0
2,1,3,3.5
3,1,4,3.0
4,1,5,4.0


In [9]:
lowest_rating = ratings['rating'].min()
highest_rating = ratings['rating'].max()
lowest_rating,highest_rating

(np.float64(0.5), np.float64(4.0))

In [10]:
reader =surprise.Reader(rating_scale=(lowest_rating, highest_rating))

data = surprise.Dataset.load_from_df(ratings,reader)
type(data)

surprise.dataset.DatasetAutoFolds

In [11]:
similarity_options = {'name':'cosine', 'user_based':True}
algo = surprise.KNNBasic(sim_options=similarity_options)
output = algo.fit(data.build_full_trainset())
# calculate the expected ratings for all the users

Computing the cosine similarity matrix...
Done computing similarity matrix.


In [12]:
from surprise.model_selection import GridSearchCV
from surprise.model_selection.split import KFold
from surprise.model_selection import train_test_split

In [13]:
trainset , testset = train_test_split(data, test_size = 0.25, random_state = 26)
similarity_options = {'name':'cosine' , 'user_based':True}
algo = surprise.KNNBasic(sim_options=similarity_options, k= 70)
algo.fit(trainset)
predictions = algo.test(testset)
surprise.accuracy.rmse(predictions)

Computing the cosine similarity matrix...
Done computing similarity matrix.
RMSE: 0.9282


np.float64(0.9282056050129103)

## Tuning for best k



In [14]:
param_grid = {'k':[25,40,50,65,75,85], 'user_based':[False, True]}
kfold = KFold(n_splits=5, random_state=26, shuffle=True)

# gs  = GridSearchCV(surprise.KNNBasic, param_grid, measures=['rmse','mae'], cv = kfold)
gs  = GridSearchCV(surprise.KNNWithZScore, param_grid, measures=['rmse','mae'], cv = kfold)
gs.fit(data)

Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computi

In [21]:
print(gs.best_score['rmse'])
print(gs.best_params['rmse'])

0.82858464566139
{'k': 85, 'user_based': False}


In [22]:
# building best model on whole data

In [23]:
algo = gs.best_estimator['rmse']
algo.fit(data.build_full_trainset())

Computing the msd similarity matrix...
Done computing similarity matrix.


Total number of items

In [24]:
iids = ratings['iid'].unique()
print(len(iids))

2071


 consider any user

In [25]:
a = [2,4,5,7,8,9]
b = [4,7,9]
np.setdiff1d(a,b)

array([2, 5, 8])

In [27]:
user = 50
u_iid = ratings[ratings['uid']==user]['iid'].unique()
print('List of items rated by user: ',u_iid)
print("\nNo of items rated by user {0}: {1} ".format(user, len(u_iid)))

iids_to_predict = np.setdiff1d(iids, u_iid)

print("\nItems not rated by the user or those items for which the expected ratings are to be predict {0}".format(iids_to_predict))

List of items rated by user:  [  8 211   3   2 219 234  12 254 250 207  11 253 236  84  10   7 233  13
   1   5   6 252 241 216 257 206   4 217   9 215 213  17 255 220 121 245
 239 251 235]

No of items rated by user 50: 39 

Items not rated by the user or those items for which the expected ratings are to be predict [  14   15   16 ... 2069 2070 2071]


In [28]:
testset = [[user, iid, 0.] for iid in iids_to_predict]
predictions = algo.test(testset)

exp_ratings = [(predictions[i].iid, predictions[i].est) for i in range(0, len(predictions))]
exp_ratings = pd.DataFrame(exp_ratings, columns=['iid', 'est_rating'])

# Sort the DataFrame instead of the list
exp_ratings.sort_values(by='est_rating', ascending=False).head(30)

,iid,est_rating
12,27,4.0
1119,1159,4.0
1932,1972,4.0
1933,1973,4.0
1149,1189,4.0
1323,1363,4.0
424,464,4.0
439,479,4.0
1259,1299,4.0
1544,1584,4.0


In [30]:
os.chdir(r'E:\PGCP-AI\PML\Cases\SURPRISE\ml-100k') 

In [37]:
movies = pd.read_csv('movies_list.csv', encoding = 'latin-1')
movies.columns = movies.columns.str.strip()
movies.columns

Index(['movie id', 'movie title', 'release date', 'video release date',
       'IMDb URL', 'unknown', 'Action', 'Adventure', 'Animation', 'Children's',
       'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir',
       'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War',
       'Western'],
      dtype='object')

In [38]:
exp_ratings = exp_ratings.merge(
    movies[['movie id', 'movie title', 'release date']],
    how='left',
    left_on='iid',
    right_on='movie id'
)
exp_ratings.sort_values(by='est_rating', ascending=False).head(10)

,iid,est_rating,movie id,movie title,release date,movie id,movie title,release date
12,27,4.0,27.0,Bad Boys (1995),01-Jan-95,27.0,Bad Boys (1995),01-Jan-95
1119,1159,4.0,1159.0,Stalker (1979),01-Jan-79,1159.0,Stalker (1979),01-Jan-79
1932,1972,4.0,NaN,NaN,NaN,NaN,NaN,NaN
1933,1973,4.0,NaN,NaN,NaN,NaN,NaN,NaN
1149,1189,4.0,1189.0,Prefontaine (1997),24-Jan-97,1189.0,Prefontaine (1997),24-Jan-97
1323,1363,4.0,1363.0,"Leopard Son, The (1996)",27-Sep-96,1363.0,"Leopard Son, The (1996)",27-Sep-96
424,464,4.0,464.0,Vanya on 42nd Street (1994),01-Jan-94,464.0,Vanya on 42nd Street (1994),01-Jan-94
439,479,4.0,479.0,Vertigo (1958),01-Jan-58,479.0,Vertigo (1958),01-Jan-58
1259,1299,4.0,1299.0,Penny Serenade (1941),01-Jan-41,1299.0,Penny Serenade (1941),01-Jan-41
1544,1584,4.0,1584.0,"Symphonie pastorale, La (1946)",01-Jan-46,1584.0,"Symphonie pastorale, La (1946)",01-Jan-46
